# Bab 16: Jaringan Saraf Tiruan

Notebook pendamping buku *Pengenalan Pola dan Machine Learning: Pendekatan Probabilistik dengan Python*.

Jalankan sel dari atas ke bawah (**Runtime → Run all** di Colab). Sel pertama menyiapkan folder kode dan modul data sintetis; setiap sel berikutnya adalah satu listing dari buku, dengan nomor penanda `# (1)`, `# (2)`, … yang dijelaskan di bawah listing pada buku. Angka keluaran bisa sedikit berbeda dari buku bila versi pustakanya berbeda (lihat Lampiran A).

In [ ]:
# Menyiapkan folder kode/ dan modul pendukung bab ini
import os, pathlib
BERKAS = {
"bab16/data_aksara.py": "\"\"\"Citra tulisan tangan sintetis 20 x 20 piksel untuk 10 kelas glif (ilustrasi).\n\nSetiap glif dibangun dari goresan sederhana (lengkung, lingkaran kecil, garis\ntegak dan mendatar) yang meniru gaya goresan aksara Jawa secara skematis.\nBentuknya TIDAK akurat sebagai aksara; untuk penelitian, gunakan himpunan data\ntulisan tangan aksara Jawa yang sebenarnya.\n\"\"\"\nimport numpy as np\nfrom PIL import Image, ImageDraw\n\ndef _busur(cx, cy, r, a0, a1, n=12):\n    t = np.linspace(np.radians(a0), np.radians(a1), n)\n    return list(zip(cx + r * np.cos(t), cy + r * np.sin(t)))\n\ndef _glif():\n    G = []\n    G.append([_busur(.3, .5, .18, 180, 360) + [(.48, .8)], [(.48, .5), (.7, .5)], _busur(.7, .5, .15, 180, 360)])\n    G.append([[(.15, .7), (.15, .35)], _busur(.32, .35, .17, 180, 360), [(.49, .35), (.49, .7)], _busur(.66, .35, .17, 180, 360) + [(.83, .7)]])\n    G.append([_busur(.5, .4, .25, 180, 360), [(.25, .4), (.25, .75), (.75, .75), (.75, .4)], _busur(.5, .55, .07, 0, 360)])\n    G.append([[(.2, .3), (.8, .3)], [(.35, .3), (.35, .8)], [(.65, .3), (.65, .8)], _busur(.5, .8, .15, 0, 180)])\n    G.append([_busur(.35, .5, .2, 90, 450), [(.55, .5), (.85, .5)], [(.85, .25), (.85, .8)]])\n    G.append([_busur(.5, .35, .25, 180, 360), [(.25, .35), (.4, .8)], [(.75, .35), (.6, .8)], [(.4, .8), (.6, .8)]])\n    G.append([[(.2, .25), (.2, .75)], [(.2, .5), (.8, .5)], [(.8, .25), (.8, .75)], _busur(.5, .3, .1, 0, 360)])\n    G.append([_busur(.3, .4, .15, 0, 360), _busur(.7, .4, .15, 0, 360), [(.15, .75), (.85, .75)]])\n    G.append([[(.15, .3), (.35, .75), (.5, .4), (.65, .75), (.85, .3)]])\n    G.append([_busur(.5, .5, .3, 200, 520), [(.5, .2), (.5, .8)]])\n    return G\n\nGLIF = _glif()\n\ndef gambar(k, rng, ukuran=20):\n    S = 64; im = Image.new(\"L\", (S, S), 0); d = ImageDraw.Draw(im)\n    a = np.radians(rng.normal(0, 12)); sk = rng.normal(1, 0.12, 2); sh = rng.normal(0, 0.2)\n    M = np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]]) @ np.array([[sk[0], sh], [0, sk[1]]])\n    geser = rng.normal(0, 0.06, 2)\n    for gores in GLIF[k]:\n        P = (np.array(gores) - 0.5 + rng.normal(0, 0.015, (len(gores), 2))) @ M.T + 0.5 + geser\n        d.line([tuple(p) for p in P * S], fill=255, width=int(rng.integers(3, 6)), joint=\"curve\")\n    x = np.asarray(im.resize((ukuran, ukuran), Image.BILINEAR), float) / 255\n    return np.clip(x + rng.normal(0, 0.1, x.shape), 0, 1)\n\ndef buat_data(n=6000, seed=0):\n    rng = np.random.default_rng(seed)\n    y = rng.integers(0, 10, n)\n    X = np.stack([gambar(k, rng) for k in y]).reshape(n, -1)\n    return X.astype(np.float32), y\n",
"bab16/mdn_torch.py": "import numpy as np\nimport torch\nfrom torch import nn\n\n# Masalah invers (ilustrasi): t -> x mudah, x -> t bercabang.\nrng = np.random.default_rng(0)\nt = rng.uniform(0, 1, 1000)\nx = t + 0.3 * np.sin(2 * np.pi * t) + rng.uniform(-0.1, 0.1, 1000)\nX, T = torch.tensor(x[:, None], dtype=torch.float32), torch.tensor(t[:, None], dtype=torch.float32)\n\nK = 3\nclass MDN(nn.Module):                                         #<1>\n    def __init__(self):\n        super().__init__()\n        self.badan = nn.Sequential(nn.Linear(1, 32), nn.Tanh(), nn.Linear(32, 32), nn.Tanh())\n        self.pi, self.mu, self.ls = nn.Linear(32, K), nn.Linear(32, K), nn.Linear(32, K)\n    def forward(self, x):\n        h = self.badan(x)\n        return torch.log_softmax(self.pi(h), 1), self.mu(h), self.ls(h)  # log pi, mu, log sigma\n\ndef nll(lpi, mu, ls, t):                                       # -log-likelihood campuran  #<2>\n    lp = lpi - ls - 0.5 * ((t - mu) / ls.exp()) ** 2 - 0.5 * np.log(2 * np.pi)\n    return -torch.logsumexp(lp, 1).mean()\n\ntorch.manual_seed(0)\nmdn, mlp = MDN(), nn.Sequential(nn.Linear(1, 32), nn.Tanh(), nn.Linear(32, 32), nn.Tanh(), nn.Linear(32, 1))\no1, o2 = torch.optim.Adam(mdn.parameters(), 3e-3), torch.optim.Adam(mlp.parameters(), 3e-3)\nfor it in range(4000):\n    o1.zero_grad(); nll(*mdn(X), T).backward(); o1.step()\n    o2.zero_grad(); ((mlp(X) - T) ** 2).mean().backward(); o2.step()   # kuadrat terkecil  #<3>\nwith torch.no_grad():\n    print(f\"-log-lik. per titik MDN: {nll(*mdn(X), T).item():.3f}\")\n    for x0 in [0.2, 0.5, 0.8]:                                 #<4>\n        lpi, mu, ls = mdn(torch.tensor([[x0]]))\n        kom = sorted(zip(lpi.exp()[0].tolist(), mu[0].tolist(), ls.exp()[0].tolist()), reverse=True)\n        print(f\"x = {x0}: kuadrat terkecil t = {mlp(torch.tensor([[x0]])).item():.2f}\")\n        print(\"   MDN: \" + \" + \".join(f\"{p:.2f} N({m:.2f}, {s:.2f})\" for p, m, s in kom if p > 0.05))\n",
"bab16/mlp_numpy.py": "import numpy as np\nfrom data_aksara import buat_data\n\nX, y = buat_data()\nXl, yl, Xu, yu = X[:5000], y[:5000], X[5000:], y[5000:]\nrng = np.random.default_rng(0)\nD, H, K = 400, 64, 10\nP = {\"W1\": rng.normal(0, np.sqrt(2 / D), (D, H)), \"b1\": np.zeros(H),   # inisialisasi He  #<1>\n     \"W2\": rng.normal(0, np.sqrt(1 / H), (H, K)), \"b2\": np.zeros(K)}\n\ndef maju(P, X):\n    a1 = X @ P[\"W1\"] + P[\"b1\"]; z1 = np.maximum(a1, 0)          # ReLU\n    a2 = z1 @ P[\"W2\"] + P[\"b2\"]\n    a2 = a2 - a2.max(1, keepdims=True)                            # softmax stabil\n    p = np.exp(a2); p /= p.sum(1, keepdims=True)\n    return a1, z1, p\n\ndef loss_dan_gradien(P, X, y):\n    a1, z1, p = maju(P, X); N = len(y)\n    loss = -np.log(p[np.arange(N), y] + 1e-12).mean()             # -log-likelihood  #<2>\n    d2 = p.copy(); d2[np.arange(N), y] -= 1; d2 /= N              # delta keluaran = p - t  #<3>\n    d1 = (d2 @ P[\"W2\"].T) * (a1 > 0)                              # propagasi balik  #<4>\n    return loss, {\"W2\": z1.T @ d2, \"b2\": d2.sum(0), \"W1\": X.T @ d1, \"b1\": d1.sum(0)}\n\n# periksa gradien dengan beda hingga pada satu bobot  #<5>\n_, g = loss_dan_gradien(P, Xl[:50], yl[:50]); e = 1e-5\nPp = {k: v.copy() for k, v in P.items()}; Pp[\"W1\"][10, 3] += e\nPm = {k: v.copy() for k, v in P.items()}; Pm[\"W1\"][10, 3] -= e\nnum = (loss_dan_gradien(Pp, Xl[:50], yl[:50])[0] - loss_dan_gradien(Pm, Xl[:50], yl[:50])[0]) / (2 * e)\nprint(f\"gradien backprop {g['W1'][10, 3]:+.6e}, beda hingga {num:+.6e}\")\n\nv = {k: np.zeros_like(w) for k, w in P.items()}\nfor ep in range(15):                                              # SGD + momentum  #<6>\n    for idx in np.array_split(rng.permutation(5000), 5000 // 64):\n        loss, g = loss_dan_gradien(P, Xl[idx], yl[idx])\n        for k in P:\n            v[k] = 0.9 * v[k] - 0.05 * g[k]; P[k] += v[k]\n    if ep % 3 == 2:\n        akur = (maju(P, Xu)[2].argmax(1) == yu).mean()\n        print(f\"epoch {ep + 1:2d}: loss latih {loss_dan_gradien(P, Xl, yl)[0]:.3f}, akurasi uji {akur:.3f}\")\nfrom sklearn.linear_model import LogisticRegression                # pembanding linear\nlin = LogisticRegression(max_iter=2000).fit(Xl, yl)\nprint(f\"regresi logistik (tanpa lapisan tersembunyi): akurasi uji {lin.score(Xu, yu):.3f}\")\n",
"bab16/optimisasi_torch.py": "import numpy as np\nimport torch\nfrom torch import nn\nfrom data_aksara import buat_data\n\nX, y = buat_data()\nXl, yl = torch.tensor(X[:5000]), torch.tensor(y[:5000])\nXu, yu = torch.tensor(X[5000:]), torch.tensor(y[5000:])\n\ndef jaringan():                                            # sama dengan versi NumPy  #<1>\n    torch.manual_seed(0)\n    return nn.Sequential(nn.Linear(400, 64), nn.ReLU(), nn.Linear(64, 10))\n\nhasil = {}\nfor nama, buat_opt in [(\"SGD\", lambda p: torch.optim.SGD(p, lr=0.05)),       #<2>\n                       (\"momentum\", lambda p: torch.optim.SGD(p, lr=0.05, momentum=0.9)),\n                       (\"Adam\", lambda p: torch.optim.Adam(p, lr=1e-3))]:\n    net = jaringan(); opt = buat_opt(net.parameters())\n    loss_fn = nn.CrossEntropyLoss()                        # softmax + -log-likelihood\n    kurva = []\n    for ep in range(15):\n        for idx in torch.randperm(5000, generator=torch.Generator().manual_seed(ep)).split(64):\n            opt.zero_grad()\n            loss = loss_fn(net(Xl[idx]), yl[idx])\n            loss.backward()                                # backprop otomatis  #<3>\n            opt.step()\n        with torch.no_grad():\n            kurva.append(loss_fn(net(Xl), yl).item())\n            akur = (net(Xu).argmax(1) == yu).float().mean().item()\n    hasil[nama] = kurva\n    print(f\"{nama:8s}: loss latih epoch 1 {kurva[0]:.3f}, epoch 15 {kurva[-1]:.3f},\"\n          f\" akurasi uji {akur:.3f}\")\nnp.savetxt(\"_kurva_opt.txt\", np.c_[np.arange(1, 16), *hasil.values()], fmt=\"%.4f\")\n",
"bab16/regularisasi_torch.py": "import numpy as np\nimport torch\nfrom torch import nn\nfrom data_aksara import buat_data\n\nX, y = buat_data()\nXl, yl = torch.tensor(X[:300]), torch.tensor(y[:300])          # data latih sedikit  #<1>\nXv, yv = torch.tensor(X[3000:4000]), torch.tensor(y[3000:4000])  # validasi\nXu, yu = torch.tensor(X[5000:]), torch.tensor(y[5000:])          # uji\n\ndef latih(wd=0.0, p_drop=0.0, epoch=300):\n    torch.manual_seed(0)\n    net = nn.Sequential(nn.Linear(400, 256), nn.ReLU(), nn.Dropout(p_drop),   #<2>\n                        nn.Linear(256, 256), nn.ReLU(), nn.Dropout(p_drop),\n                        nn.Linear(256, 10))\n    opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=wd)        #<3>\n    f = nn.CrossEntropyLoss(); kurva, terbaik = [], (np.inf, None, 0)\n    for ep in range(epoch):\n        net.train(); opt.zero_grad(); f(net(Xl), yl).backward(); opt.step()\n        net.eval()\n        with torch.no_grad():\n            lv = f(net(Xv), yv).item(); kurva.append(lv)\n            if lv < terbaik[0]:                                                #<4>\n                terbaik = (lv, {k: v.clone() for k, v in net.state_dict().items()}, ep)\n    with torch.no_grad():\n        ukur = lambda: (f(net(Xu), yu).item(), (net(Xu).argmax(1) == yu).float().mean().item())\n        akhir = ukur(); net.load_state_dict(terbaik[1]); dini = ukur()\n    return kurva, akhir, dini, terbaik[2]\n\nhasil = {}\nprint(\"                   epoch 300       | early stopping\")\nprint(\"                   NLL   akurasi   | epoch  NLL   akurasi\")\nfor nama, kw in [(\"tanpa regularisasi\", {}), (\"weight decay 1,0\", {\"wd\": 1.0}),\n                 (\"dropout 0,5\", {\"p_drop\": 0.5})]:\n    kurva, (n1, a1), (n2, a2), ep = latih(**kw); hasil[nama] = kurva\n    print(f\"{nama:18s} {n1:.2f}  {a1:.3f}     | {ep + 1:4d}   {n2:.2f}  {a2:.3f}\")\nnp.savetxt(\"_kurva_reg.txt\", np.c_[np.arange(1, 301), *hasil.values()], fmt=\"%.4f\")\n"
}
akar = pathlib.Path('pola-ml/kode').resolve()
for nama, isi in BERKAS.items():
    p = akar / nama; p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(isi, encoding='utf-8')
os.chdir(akar / 'bab16')
print('folder kerja:', os.getcwd())

## Listing 16.1: Jaringan dua lapis dan backpropagation dengan NumPy

Berkas: `kode/bab16/mlp_numpy.py`

In [ ]:
import numpy as np
from data_aksara import buat_data

X, y = buat_data()
Xl, yl, Xu, yu = X[:5000], y[:5000], X[5000:], y[5000:]
rng = np.random.default_rng(0)
D, H, K = 400, 64, 10
P = {"W1": rng.normal(0, np.sqrt(2 / D), (D, H)), "b1": np.zeros(H),   # inisialisasi He  # (1)
     "W2": rng.normal(0, np.sqrt(1 / H), (H, K)), "b2": np.zeros(K)}

def maju(P, X):
    a1 = X @ P["W1"] + P["b1"]; z1 = np.maximum(a1, 0)          # ReLU
    a2 = z1 @ P["W2"] + P["b2"]
    a2 = a2 - a2.max(1, keepdims=True)                            # softmax stabil
    p = np.exp(a2); p /= p.sum(1, keepdims=True)
    return a1, z1, p

def loss_dan_gradien(P, X, y):
    a1, z1, p = maju(P, X); N = len(y)
    loss = -np.log(p[np.arange(N), y] + 1e-12).mean()             # -log-likelihood  # (2)
    d2 = p.copy(); d2[np.arange(N), y] -= 1; d2 /= N              # delta keluaran = p - t  # (3)
    d1 = (d2 @ P["W2"].T) * (a1 > 0)                              # propagasi balik  # (4)
    return loss, {"W2": z1.T @ d2, "b2": d2.sum(0), "W1": X.T @ d1, "b1": d1.sum(0)}

# periksa gradien dengan beda hingga pada satu bobot  # (5)
_, g = loss_dan_gradien(P, Xl[:50], yl[:50]); e = 1e-5
Pp = {k: v.copy() for k, v in P.items()}; Pp["W1"][10, 3] += e
Pm = {k: v.copy() for k, v in P.items()}; Pm["W1"][10, 3] -= e
num = (loss_dan_gradien(Pp, Xl[:50], yl[:50])[0] - loss_dan_gradien(Pm, Xl[:50], yl[:50])[0]) / (2 * e)
print(f"gradien backprop {g['W1'][10, 3]:+.6e}, beda hingga {num:+.6e}")

v = {k: np.zeros_like(w) for k, w in P.items()}
for ep in range(15):                                              # SGD + momentum  # (6)
    for idx in np.array_split(rng.permutation(5000), 5000 // 64):
        loss, g = loss_dan_gradien(P, Xl[idx], yl[idx])
        for k in P:
            v[k] = 0.9 * v[k] - 0.05 * g[k]; P[k] += v[k]
    if ep % 3 == 2:
        akur = (maju(P, Xu)[2].argmax(1) == yu).mean()
        print(f"epoch {ep + 1:2d}: loss latih {loss_dan_gradien(P, Xl, yl)[0]:.3f}, akurasi uji {akur:.3f}")
from sklearn.linear_model import LogisticRegression                # pembanding linear
lin = LogisticRegression(max_iter=2000).fit(Xl, yl)
print(f"regresi logistik (tanpa lapisan tersembunyi): akurasi uji {lin.score(Xu, yu):.3f}")

## Listing 16.2: Jaringan yang sama dalam PyTorch, dengan tiga optimizer

Berkas: `kode/bab16/optimisasi_torch.py`

In [ ]:
import numpy as np
import torch
from torch import nn
from data_aksara import buat_data

X, y = buat_data()
Xl, yl = torch.tensor(X[:5000]), torch.tensor(y[:5000])
Xu, yu = torch.tensor(X[5000:]), torch.tensor(y[5000:])

def jaringan():                                            # sama dengan versi NumPy  # (1)
    torch.manual_seed(0)
    return nn.Sequential(nn.Linear(400, 64), nn.ReLU(), nn.Linear(64, 10))

hasil = {}
for nama, buat_opt in [("SGD", lambda p: torch.optim.SGD(p, lr=0.05)),       # (2)
                       ("momentum", lambda p: torch.optim.SGD(p, lr=0.05, momentum=0.9)),
                       ("Adam", lambda p: torch.optim.Adam(p, lr=1e-3))]:
    net = jaringan(); opt = buat_opt(net.parameters())
    loss_fn = nn.CrossEntropyLoss()                        # softmax + -log-likelihood
    kurva = []
    for ep in range(15):
        for idx in torch.randperm(5000, generator=torch.Generator().manual_seed(ep)).split(64):
            opt.zero_grad()
            loss = loss_fn(net(Xl[idx]), yl[idx])
            loss.backward()                                # backprop otomatis  # (3)
            opt.step()
        with torch.no_grad():
            kurva.append(loss_fn(net(Xl), yl).item())
            akur = (net(Xu).argmax(1) == yu).float().mean().item()
    hasil[nama] = kurva
    print(f"{nama:8s}: loss latih epoch 1 {kurva[0]:.3f}, epoch 15 {kurva[-1]:.3f},"
          f" akurasi uji {akur:.3f}")
np.savetxt("_kurva_opt.txt", np.c_[np.arange(1, 16), *hasil.values()], fmt="%.4f")

## Listing 16.3: Weight decay, dropout, dan early stopping pada data latih kecil

Berkas: `kode/bab16/regularisasi_torch.py`

In [ ]:
import numpy as np
import torch
from torch import nn
from data_aksara import buat_data

X, y = buat_data()
Xl, yl = torch.tensor(X[:300]), torch.tensor(y[:300])          # data latih sedikit  # (1)
Xv, yv = torch.tensor(X[3000:4000]), torch.tensor(y[3000:4000])  # validasi
Xu, yu = torch.tensor(X[5000:]), torch.tensor(y[5000:])          # uji

def latih(wd=0.0, p_drop=0.0, epoch=300):
    torch.manual_seed(0)
    net = nn.Sequential(nn.Linear(400, 256), nn.ReLU(), nn.Dropout(p_drop),   # (2)
                        nn.Linear(256, 256), nn.ReLU(), nn.Dropout(p_drop),
                        nn.Linear(256, 10))
    opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=wd)        # (3)
    f = nn.CrossEntropyLoss(); kurva, terbaik = [], (np.inf, None, 0)
    for ep in range(epoch):
        net.train(); opt.zero_grad(); f(net(Xl), yl).backward(); opt.step()
        net.eval()
        with torch.no_grad():
            lv = f(net(Xv), yv).item(); kurva.append(lv)
            if lv < terbaik[0]:                                                # (4)
                terbaik = (lv, {k: v.clone() for k, v in net.state_dict().items()}, ep)
    with torch.no_grad():
        ukur = lambda: (f(net(Xu), yu).item(), (net(Xu).argmax(1) == yu).float().mean().item())
        akhir = ukur(); net.load_state_dict(terbaik[1]); dini = ukur()
    return kurva, akhir, dini, terbaik[2]

hasil = {}
print("                   epoch 300       | early stopping")
print("                   NLL   akurasi   | epoch  NLL   akurasi")
for nama, kw in [("tanpa regularisasi", {}), ("weight decay 1,0", {"wd": 1.0}),
                 ("dropout 0,5", {"p_drop": 0.5})]:
    kurva, (n1, a1), (n2, a2), ep = latih(**kw); hasil[nama] = kurva
    print(f"{nama:18s} {n1:.2f}  {a1:.3f}     | {ep + 1:4d}   {n2:.2f}  {a2:.3f}")
np.savetxt("_kurva_reg.txt", np.c_[np.arange(1, 301), *hasil.values()], fmt="%.4f")

## Listing 16.4: Mixture density network untuk masalah invers

Berkas: `kode/bab16/mdn_torch.py`

In [ ]:
import numpy as np
import torch
from torch import nn

# Masalah invers (ilustrasi): t -> x mudah, x -> t bercabang.
rng = np.random.default_rng(0)
t = rng.uniform(0, 1, 1000)
x = t + 0.3 * np.sin(2 * np.pi * t) + rng.uniform(-0.1, 0.1, 1000)
X, T = torch.tensor(x[:, None], dtype=torch.float32), torch.tensor(t[:, None], dtype=torch.float32)

K = 3
class MDN(nn.Module):                                         # (1)
    def __init__(self):
        super().__init__()
        self.badan = nn.Sequential(nn.Linear(1, 32), nn.Tanh(), nn.Linear(32, 32), nn.Tanh())
        self.pi, self.mu, self.ls = nn.Linear(32, K), nn.Linear(32, K), nn.Linear(32, K)
    def forward(self, x):
        h = self.badan(x)
        return torch.log_softmax(self.pi(h), 1), self.mu(h), self.ls(h)  # log pi, mu, log sigma

def nll(lpi, mu, ls, t):                                       # -log-likelihood campuran  # (2)
    lp = lpi - ls - 0.5 * ((t - mu) / ls.exp()) ** 2 - 0.5 * np.log(2 * np.pi)
    return -torch.logsumexp(lp, 1).mean()

torch.manual_seed(0)
mdn, mlp = MDN(), nn.Sequential(nn.Linear(1, 32), nn.Tanh(), nn.Linear(32, 32), nn.Tanh(), nn.Linear(32, 1))
o1, o2 = torch.optim.Adam(mdn.parameters(), 3e-3), torch.optim.Adam(mlp.parameters(), 3e-3)
for it in range(4000):
    o1.zero_grad(); nll(*mdn(X), T).backward(); o1.step()
    o2.zero_grad(); ((mlp(X) - T) ** 2).mean().backward(); o2.step()   # kuadrat terkecil  # (3)
with torch.no_grad():
    print(f"-log-lik. per titik MDN: {nll(*mdn(X), T).item():.3f}")
    for x0 in [0.2, 0.5, 0.8]:                                 # (4)
        lpi, mu, ls = mdn(torch.tensor([[x0]]))
        kom = sorted(zip(lpi.exp()[0].tolist(), mu[0].tolist(), ls.exp()[0].tolist()), reverse=True)
        print(f"x = {x0}: kuadrat terkecil t = {mlp(torch.tensor([[x0]])).item():.2f}")
        print("   MDN: " + " + ".join(f"{p:.2f} N({m:.2f}, {s:.2f})" for p, m, s in kom if p > 0.05))